# NYC Yellow Taxi Trip Data — Cleaning Pipeline

**Dataset:** `yellow_tripdata_2025-11.parquet` (4.18M rows, 20 columns)

This notebook follows a structured cleaning pipeline:
1. Load & Inspect
2. Understand & Handle Missing Data
3. Engineer Key Columns
4. Remove Invalid Trips (datetime & speed)
5. Flag Negative Monetary Values
6. Decode Categorical IDs
7. Final Clean Dataset

## Step 1 — Load & Inspect

In [2]:
import pandas as pd
import numpy as np

df = pd.read_parquet("yellow_tripdata_2025-11.parquet")

print(f"Shape: {df.shape}")
print(f"\nDtypes:")
df.info()
df.head()

Shape: (4181444, 20)

Dtypes:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4181444 entries, 0 to 4181443
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64     

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2025-11-01 00:13:25,2025-11-01 00:13:25,1.0,1.68,1.0,N,43,186,1,14.9,0.00,0.5,1.50,0.00,1.0,22.15,2.5,0.00,0.75
1,2,2025-11-01 00:49:07,2025-11-01 01:01:22,1.0,2.28,1.0,N,142,237,1,14.2,1.00,0.5,4.99,0.00,1.0,24.94,2.5,0.00,0.75
2,1,2025-11-01 00:07:19,2025-11-01 00:20:41,0.0,2.70,1.0,N,163,238,1,15.6,4.25,0.5,4.27,0.00,1.0,25.62,2.5,0.00,0.75
3,2,2025-11-01 00:00:00,2025-11-01 01:01:03,3.0,12.87,1.0,N,138,261,1,66.7,6.00,0.5,0.00,6.94,1.0,86.14,2.5,1.75,0.75
4,1,2025-11-01 00:18:50,2025-11-01 00:49:32,0.0,8.40,1.0,N,138,37,2,39.4,7.75,0.5,0.00,0.00,1.0,48.65,0.0,1.75,0.00


In [3]:
# Quick statistical summary — look for suspicious min/max values
df.describe()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
count,4.181444e+06,4181444,4181444,3.166704e+06,4.181444e+06,3.166704e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,4.181444e+06,3.166704e+06,3.166704e+06,4.181444e+06
mean,1.879423e+00,2025-11-15 10:46:58.380460,2025-11-15 11:05:20.075878,1.287101e+00,6.530914e+00,4.464459e+00,1.620342e+02,1.615968e+02,9.050739e-01,1.711793e+01,1.059649e+00,4.818083e-01,2.862241e+00,5.179058e-01,9.498621e-01,2.561495e+01,2.185029e+00,1.430022e-01,5.350028e-01
min,1.000000e+00,2008-12-31 23:04:21,2008-12-31 23:32:25,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00,-1.508700e+03,-8.500000e+00,-5.000000e-01,-3.333300e+02,-1.150500e+02,-1.000000e+00,-1.514450e+03,-2.500000e+00,-1.750000e+00,-7.500000e-01
25%,2.000000e+00,2025-11-08 00:02:02.750000,2025-11-08 00:17:31.750000,1.000000e+00,1.050000e+00,1.000000e+00,1.140000e+02,1.070000e+02,1.000000e+00,7.900000e+00,0.000000e+00,5.000000e-01,0.000000e+00,0.000000e+00,1.000000e+00,1.435000e+01,2.500000e+00,0.000000e+00,0.000000e+00
50%,2.000000e+00,2025-11-15 10:54:55,2025-11-15 11:08:58.500000,1.000000e+00,1.890000e+00,1.000000e+00,1.610000e+02,1.620000e+02,1.000000e+00,1.280000e+01,0.000000e+00,5.000000e-01,2.000000e+00,0.000000e+00,1.000000e+00,2.058000e+01,2.500000e+00,0.000000e+00,7.500000e-01
75%,2.000000e+00,2025-11-22 11:27:26.250000,2025-11-22 11:42:58.250000,1.000000e+00,3.840000e+00,1.000000e+00,2.330000e+02,2.340000e+02,1.000000e+00,2.190000e+01,2.500000e+00,5.000000e-01,4.000000e+00,0.000000e+00,1.000000e+00,3.005000e+01,2.500000e+00,0.000000e+00,7.500000e-01
max,7.000000e+00,2025-11-30 23:59:59,2025-12-01 21:41:00,8.000000e+00,2.560676e+05,9.900000e+01,2.650000e+02,2.650000e+02,4.000000e+00,1.508700e+03,1.500000e+01,1.050000e+01,5.750000e+02,3.000000e+02,2.500000e+00,1.514450e+03,2.500000e+00,6.750000e+00,7.500000e-01
std,7.463690e-01,NaN,NaN,7.043416e-01,6.274616e+02,1.782839e+01,6.633875e+01,7.054632e+01,6.988073e-01,1.957928e+01,1.736080e+00,1.200647e-01,4.002382e+00,2.140548e+00,2.656246e-01,2.377465e+01,9.159368e-01,5.190179e-01,3.516708e-01


## Step 2 — Understand Missing Data

Five columns have **~24% missing** values: `passenger_count`, `RatecodeID`, `store_and_fwd_flag`, `congestion_surcharge`, `Airport_fee`.

The key question is: **why are they missing?**

In [4]:
# Overall missing value counts
missing = df.isna().sum()
missing[missing > 0]

passenger_count         1014740
RatecodeID              1014740
store_and_fwd_flag      1014740
congestion_surcharge    1014740
Airport_fee             1014740
dtype: int64

In [5]:
# Are the missing values tied to specific vendors?
# If missingness is vendor-specific, it's a reporting gap — NOT random data corruption.

cols_with_nulls = ["passenger_count", "RatecodeID", "store_and_fwd_flag", "congestion_surcharge", "Airport_fee"]

df.groupby("VendorID")[cols_with_nulls].apply(lambda x: x.isna().mean().round(3))

,passenger_count,RatecodeID,store_and_fwd_flag,congestion_surcharge,Airport_fee
VendorID,,,,,
1,0.168,0.168,0.168,0.168,0.168
2,0.265,0.265,0.265,0.265,0.265
6,1.000,1.000,1.000,1.000,1.000
7,0.000,0.000,0.000,0.000,0.000


### Findings

Missingness is **perfectly correlated with VendorID** — it is structural, not random:

| Vendor | Name | Missing Rate |
|--------|------|--------------|
| 1 | Creative Mobile Technologies | ~17% |
| 2 | Curb Mobility | ~26% |
| 6 | Myle Technologies | **100%** |
| 7 | Helix | **0%** |

**Decision:**
- **Vendor 6** never reports extended schema fields → **exclude from any analysis using those columns**
- For Vendors 1 & 2, missing rows represent a sub-fleet that doesn't transmit those fields
- **Do NOT impute** — these are structural omissions, not measurement noise
- Use a **layered approach**: keep all vendors for financial/temporal analysis; filter to vendors 1, 2, 7 for passenger/rate/surcharge analysis

## Step 3 — Engineer Key Columns

In [6]:
# Trip duration in seconds
df["trip_duration_sec"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds()

# Average speed in mph (distance in miles / duration in hours)
# Note: We compute this before filtering so we can use it as a filter criterion.
# Division by zero (0-duration trips) will produce inf — handled in Step 4.
df["avg_speed_mph"] = df["trip_distance"] / (df["trip_duration_sec"] / 3600)

print("Columns added: trip_duration_sec, avg_speed_mph")
print(f"Rows with trip_duration_sec == 0: {(df['trip_duration_sec'] == 0).sum():,}")
print(f"Rows with trip_duration_sec < 0 (dropoff before pickup): {(df['trip_duration_sec'] < 0).sum():,}")

Columns added: trip_duration_sec, avg_speed_mph
Rows with trip_duration_sec == 0: 60,685
Rows with trip_duration_sec < 0 (dropoff before pickup): 1,435


## Step 4 — Remove Invalid Trips

### 4a. Zero/negative duration trips
Trips where pickup == dropoff time (or dropoff is before pickup) are system errors or logging artifacts.

In [7]:
# NOTE: Vendor 7 (Helix) has ALL trips with duration == 0.
# This is a Helix-specific reporting issue (they log pre-booked fares, not live meter times).
# Vendor 7 will be dropped entirely by this filter — that is expected and correct.

print("Vendor 7 trip duration sample:")
print(df[df["VendorID"] == 7]["trip_duration_sec"].describe())

print(f"\nBefore filter: {df.shape[0]:,} rows")
clean_df = df[df["trip_duration_sec"] > 0].copy()
print(f"After removing zero/negative duration: {clean_df.shape[0]:,} rows")
print(clean_df["VendorID"].value_counts())

Vendor 7 trip duration sample:
count    60043.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: trip_duration_sec, dtype: float64

Before filter: 4,181,444 rows
After removing zero/negative duration: 4,119,324 rows
VendorID
2    3294654
1     820437
6       4233
Name: count, dtype: int64


### 4b. Out-of-month timestamps
This is November 2025 data. Any pickups outside Nov 2025 are erroneous records.

In [8]:
print(f"Earliest pickup: {clean_df['tpep_pickup_datetime'].min()}")
print(f"Latest pickup:   {clean_df['tpep_pickup_datetime'].max()}")

# How many rows are outside Nov 2025?
out_of_range = clean_df[
    (clean_df["tpep_pickup_datetime"] < "2025-11-01") |
    (clean_df["tpep_pickup_datetime"] >= "2025-12-01")
]
print(f"\nOut-of-range pickup rows: {len(out_of_range):,}")

# Filter to November 2025 only
clean_df = clean_df[
    (clean_df["tpep_pickup_datetime"] >= "2025-11-01") &
    (clean_df["tpep_pickup_datetime"] < "2025-12-01")
].copy()
print(f"After date filter: {clean_df.shape[0]:,} rows")

Earliest pickup: 2008-12-31 23:04:21
Latest pickup:   2025-11-30 23:59:59

Out-of-range pickup rows: 24
After date filter: 4,119,300 rows


### 4c. Unrealistic speed
After removing zero-duration trips, `avg_speed_mph` can still be:
- **inf** — trip distance > 0 but duration is tiny (nearly instantaneous)
- **Extremely high** — GPS/odometer errors  
- **Zero** — legitimate slow trips (traffic) or 0-distance trips (cancelled?)

We keep trips with speed between **1 mph and 200 mph** as a reasonable NYC taxi range.

In [9]:
print(f"Speed distribution:")
print(clean_df["avg_speed_mph"].replace([np.inf, -np.inf], np.nan).describe())
print(f"\nInf speed rows: {np.isinf(clean_df['avg_speed_mph']).sum():,}")
print(f"Speed > 200 mph rows: {(clean_df['avg_speed_mph'] > 200).sum():,}")

before = len(clean_df)
clean_df = clean_df[
    (clean_df["avg_speed_mph"] > 1) &
    (clean_df["avg_speed_mph"] < 200)
].copy()
print(f"\nRows removed by speed filter: {before - len(clean_df):,}")
print(f"After speed filter: {clean_df.shape[0]:,} rows")

Speed distribution:
count    4.119300e+06
mean     2.247615e+01
std      2.812503e+03
min      0.000000e+00
25%      6.352941e+00
50%      8.954622e+00
75%      1.266184e+01
max      1.897379e+06
Name: avg_speed_mph, dtype: float64

Inf speed rows: 0
Speed > 200 mph rows: 643

Rows removed by speed filter: 130,177
After speed filter: 3,989,123 rows


## Step 5 — Handle Negative Monetary Values

Negative amounts exist across `fare_amount`, `total_amount`, and other fee columns — **predominantly in Vendor 2**.

**Decision: Do NOT blindly drop them.**
Negative values across all payment types (including Cash and Credit Card) indicate financial reversals or accounting adjustments — not data errors. Dropping them would distort revenue calculations.

Instead: **flag them** and let downstream analysis decide.

In [10]:
# Check the scale of negative monetary records
money_cols = [
    "fare_amount", "extra", "mta_tax", "tip_amount",
    "tolls_amount", "improvement_surcharge",
    "total_amount", "congestion_surcharge",
    "Airport_fee", "cbd_congestion_fee"
]

print("Minimum values by vendor (shows where negatives live):")
clean_df.groupby("VendorID")[money_cols].min()

Minimum values by vendor (shows where negatives live):


,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
VendorID,,,,,,,,,,
1,0.0,-0.75,0.0,0.00,0.00,0.0,0.00,0.0,0.00,0.00
2,-1245.5,-7.50,-0.5,-333.33,-115.05,-1.0,-1261.31,-2.5,-1.75,-0.75
6,0.0,0.00,0.5,0.00,0.00,0.3,0.00,NaN,NaN,0.00


In [11]:
# Flag rows that appear to be reversals/adjustments
# A negative total_amount is the clearest signal of a financial reversal.
clean_df["is_refund_or_adjustment"] = clean_df["total_amount"] < 0

print(f"Flagged as refund/adjustment: {clean_df['is_refund_or_adjustment'].sum():,} rows")
print(f"({clean_df['is_refund_or_adjustment'].mean():.1%} of dataset)")

# For revenue analysis, use only positive-total trips:
# revenue_df = clean_df[~clean_df["is_refund_or_adjustment"]]

Flagged as refund/adjustment: 115,115 rows
(2.9% of dataset)


## Step 6 — Decode Categorical IDs

In [12]:
# Decode payment_type
payment_map = {
    0: "Flex Fare",
    1: "Credit Card",
    2: "Cash",
    3: "No Charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided"
}
clean_df["payment_type_label"] = clean_df["payment_type"].map(payment_map)

# Decode VendorID
vendor_map = {
    1: "Creative Mobile Technologies",
    2: "Curb Mobility",
    6: "Myle Technologies",
    7: "Helix"
}
clean_df["vendor_name"] = clean_df["VendorID"].map(vendor_map)

# Decode RatecodeID (only where not null)
rate_code_map = {
    1.0: "Standard rate",
    2.0: "JFK",
    3.0: "Newark",
    4.0: "Nassau or Westchester",
    5.0: "Negotiated fare",
    6.0: "Group ride",
    99.0: "Null/unknown"
}
clean_df["rate_code_label"] = clean_df["RatecodeID"].map(rate_code_map)

print("Labels added: payment_type_label, vendor_name, rate_code_label")

Labels added: payment_type_label, vendor_name, rate_code_label


## Step 7 — Final Clean Dataset Summary

In [13]:
print("=" * 55)
print("FINAL CLEAN DATASET")
print("=" * 55)
print(f"Rows:    {clean_df.shape[0]:,}")
print(f"Columns: {clean_df.shape[1]}")
print(f"\nRows removed from original {len(df):,}:")
print(f"  Zero/negative duration:  {(df['trip_duration_sec'] <= 0).sum():,}")
print(f"  Out-of-month pickups:    (included in duration filter for Vendor 7)")
print(f"  Unrealistic speed:       filtered in Step 4c")
print(f"\nVendor breakdown:")
print(clean_df['VendorID'].value_counts())

print(f"\nRemaining nulls:")
nulls = clean_df.isna().sum()
print(nulls[nulls > 0])

print("\nNote: Nulls in passenger_count, RatecodeID, congestion_surcharge,")
print("Airport_fee are structural (vendor reporting gaps) — not errors.")

FINAL CLEAN DATASET
Rows:    3,989,123
Columns: 26

Rows removed from original 4,181,444:
  Zero/negative duration:  62,120
  Out-of-month pickups:    (included in duration filter for Vendor 7)
  Unrealistic speed:       filtered in Step 4c

Vendor breakdown:
VendorID
2    3176086
1     808813
6       4224
Name: count, dtype: int64

Remaining nulls:
passenger_count         928828
RatecodeID              928828
store_and_fwd_flag      928828
congestion_surcharge    928828
Airport_fee             928828
rate_code_label         928828
dtype: int64

Note: Nulls in passenger_count, RatecodeID, congestion_surcharge,
Airport_fee are structural (vendor reporting gaps) — not errors.


In [14]:
# ============================================================
# LAYERED ANALYSIS REFERENCE
# Use the appropriate subset depending on your question:
# ============================================================

# LAYER 1 — All vendors, no missing data issues
# Use for: demand, revenue, time patterns, speed analysis
# Columns: datetimes, trip_distance, all fare/fee columns, PU/DOLocationID
layer1_df = clean_df.copy()

# LAYER 2 — Extended schema (Vendors 1 & 2 only, drop nulls)
# Use for: passenger counts, rate codes, congestion/airport fees
layer2_df = clean_df[
    clean_df["VendorID"].isin([1, 2])
].dropna(subset=["passenger_count", "RatecodeID", "congestion_surcharge", "Airport_fee"]).copy()

# REVENUE ANALYSIS — exclude refunds/adjustments
revenue_df = clean_df[~clean_df["is_refund_or_adjustment"]].copy()

print(f"Layer 1 (all vendors): {len(layer1_df):,} rows")
print(f"Layer 2 (extended schema, vendors 1+2): {len(layer2_df):,} rows")
print(f"Revenue analysis (no refunds): {len(revenue_df):,} rows")

Layer 1 (all vendors): 3,989,123 rows
Layer 2 (extended schema, vendors 1+2): 3,060,295 rows
Revenue analysis (no refunds): 3,874,008 rows


In [15]:
# Save cleaned dataset
clean_df.to_parquet("yellow_tripdata_2025-11_clean.parquet", index=False)
print("Saved: yellow_tripdata_2025-11_clean.parquet")

Saved: yellow_tripdata_2025-11_clean.parquet
